# Hyperbolic Activation Function (Eq. 34) and Connection Weight (Eq.2)

This notebook establishes the forward activation mechanics needed before the energy formulation is introduced.

## Step 0 — formula and data

Eq. (34) defines the hyperbolic activation:

$$
g(z) = \frac{z}{|z|} = \frac{x + u y}{\sqrt{x^2 - y^2}} \qquad \text{(Eq. 34)}
$$

This formula comes from the HHNN paper and maps an input in the admissible hyperbolic domain onto the unit hyperbola. A related normalization appears in the HBM paper:

$$
f_1(I) = \frac{I}{|I|} \qquad \text{(Eq. 11)}
$$


This is the mapping from MovieLens signals into a hyperbolic number \(z = x + u y\), so we can compute the activation step by step.

In [41]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve project root regardless of where the notebook runs
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data"
print("project_root:", project_root)
print("data_dir:", data_dir)


project_root: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys
data_dir: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data


## Step 1 — load hyperbolic visible states

Notebook 01 converts user–movie preferences into hyperbolic-valued visible states. Each state can be written as

$$
z = x + u y \qquad \text{(Eq. 12–15)}
$$

and in exponential form as

$$
z = \exp(u t) = \cosh(t) + u\sinh(t) \qquad \text{(Eq. 3)}
$$

These states now play the role of visible-layer inputs for the next computation stage.

In [42]:
# Load hyperbolic visible states from Notebook 01 outputs
processed_dir = data_dir / "processed"
binary_path = processed_dir / "rbm_input_binary.csv"

if not binary_path.exists():
    raise FileNotFoundError(
        "rbm_input_binary.csv not found. Run notebook 01 to generate it."
    )

rbm_input = pd.read_csv(binary_path, index_col=0)

# Use one user's visible vector as a compact example
user_id = rbm_input.index[0]
v_visible = rbm_input.loc[user_id].values.astype(int)

# Hyperbolic embedding via Eq. (3) with t0
# z = exp(u t) = cosh(t) + u sinh(t)
t0 = 0.8
x_visible = np.cosh(t0) * np.ones_like(v_visible, dtype=float)
y_visible = np.where(v_visible == 1, np.sinh(t0), -np.sinh(t0)).astype(float)

z_visible = {
    "x": x_visible,
    "y": y_visible,
}

print("[Step 1] Loaded binary visible vector for user:", user_id)
print("v_visible shape:", v_visible.shape)
print("z_visible x shape:", z_visible["x"].shape)
print("z_visible y shape:", z_visible["y"].shape)
print("v_visible (first 12):", v_visible[:12])
print("z_visible x (first 12):", z_visible["x"][:12])
print("z_visible y (first 12):", z_visible["y"][:12])

preview_df = pd.DataFrame({
    "v_visible": v_visible[:12],
    "x_visible": z_visible["x"][:12],
    "y_visible": z_visible["y"][:12],
})
print("[Step 1] Preview table:")
display(preview_df)


[Step 1] Loaded binary visible vector for user: 68
v_visible shape: (1320,)
z_visible x shape: (1320,)
z_visible y shape: (1320,)
v_visible (first 12): [0 0 0 0 0 0 0 0 0 1 0 0]
z_visible x (first 12): [1.33743495 1.33743495 1.33743495 1.33743495 1.33743495 1.33743495
 1.33743495 1.33743495 1.33743495 1.33743495 1.33743495 1.33743495]
z_visible y (first 12): [-0.88810598 -0.88810598 -0.88810598 -0.88810598 -0.88810598 -0.88810598
 -0.88810598 -0.88810598 -0.88810598  0.88810598 -0.88810598 -0.88810598]
[Step 1] Preview table:


,v_visible,x_visible,y_visible
0,0,1.337435,-0.888106
1,0,1.337435,-0.888106
2,0,1.337435,-0.888106
3,0,1.337435,-0.888106
4,0,1.337435,-0.888106
5,0,1.337435,-0.888106
6,0,1.337435,-0.888106
7,0,1.337435,-0.888106
8,0,1.337435,-0.888106
9,1,1.337435,0.888106


## Step 2 — define connection weights

In the HHNN paper, each neuron receives weighted hyperbolic inputs from other neurons, and the weighted sum input is defined later in Eq. (39). Here we define a bipartite RBM-style weight matrix:

$$
W \in \mathbb{R}^{n_{\text{visible}} \times n_{\text{hidden}}} \qquad \text{(Eq. 2)}
$$

- visible size = number of movies
- hidden size = number of latent preference factors
- weights connect visible movie units to hidden preference units

Note: in the HHNN setting, connection weights are constrained in the admissible hyperbolic domain and symmetry matters later for energy analysis, but in this notebook the goal is only to set up the forward hidden computation.

In [35]:
# Define dimensions
n_visible = z_visible["x"].shape[0]
n_hidden = 64

# Small random Gaussian initialization for weights
rng = np.random.default_rng(42)
W = rng.normal(0.0, 0.01, size=(n_visible, n_hidden))

# Biases (real-valued for this forward pass)
hidden_bias = np.zeros(n_hidden)
visible_bias = np.zeros(n_visible)

print("n_visible:", n_visible)
print("n_hidden:", n_hidden)
print("W shape:", W.shape)
print("hidden_bias shape:", hidden_bias.shape)
print("visible_bias shape:", visible_bias.shape)


n_visible: 1320
n_hidden: 64
W shape: (1320, 64)
hidden_bias shape: (64,)
visible_bias shape: (1320,)


## Step 3 — compute weighted sum input

Introduce the weighted sum from the HHNN paper:

$$
I_k = \sum_{j \ne k} w_{kj} z_j \qquad \text{(Eq. 39)}
$$

In this project’s bipartite RBM-style implementation, the hidden pre-activation is computed as:

$$
I^{(h)} = W^\top z_{\text{visible}} + b_{\text{hidden}} \qquad \text{(Eq. 39)}
$$

This is the weighted aggregation stage. Each hidden unit collects information from all visible movie units.

In [36]:
# Weighted sum input for hidden units (hyperbolic components)
hidden_input_x = W.T @ z_visible["x"] + hidden_bias
hidden_input_y = W.T @ z_visible["y"]

hidden_input = {
    "x": hidden_input_x,
    "y": hidden_input_y,
}

print("hidden_input x shape:", hidden_input["x"].shape)
print("hidden_input y shape:", hidden_input["y"].shape)
print("hidden_input x (first 8):", hidden_input["x"][:8])
print("hidden_input y (first 8):", hidden_input["y"][:8])


hidden_input x shape: (64,)
hidden_input y shape: (64,)
hidden_input x (first 8): [ 0.05518485 -0.59525161 -0.38449257 -0.1349206   0.72770967 -0.2877433
 -0.67020703  0.42852358]
hidden_input y (first 8): [-0.07526855  0.43272423  0.32574617  0.05519919 -0.5302737   0.17106914
  0.42040731 -0.27105999]


## Step 4 — apply the hyperbolic activation

Re-introduce the activation formula:

$$
g(z) = \frac{z}{|z|} = \frac{x + u y}{\sqrt{x^2 - y^2}} \qquad \text{(Eq. 34)}
$$

The equivalent continuous HBM form is:

$$
f_1(I) = \frac{I}{|I|} \qquad \text{(Eq. 11)}
$$

This normalization maps the weighted input back onto the valid hyperbolic state manifold (unit hyperbola). We implement the hyperbolic modulus as:

$$
|z| = \sqrt{x^2 - y^2} \qquad \text{(Eq. 34 x check the eq)}
$$

In [37]:
def hyperbolic_modulus(x, y):
    # Hyperbolic modulus |z| = sqrt(x^2 - y^2)
    return np.sqrt(np.maximum(x**2 - y**2, 1e-12))


def hyperbolic_activation(x, y):
    # g(z) = z / |z| with z = x + u y
    modulus = hyperbolic_modulus(x, y)
    gx = x / modulus
    gy = y / modulus
    return gx, gy


## Step 5 — compute hidden-state output

Apply the activation to the weighted input and compute the hidden representation:

$$
z^{(h)} = g\!\left(I^{(h)}\right) \qquad \text{(Eq. 34)}
$$

- this is the hidden-layer response
- in recommender language, these hidden units represent latent taste factors
- this is the first bridge from hyperbolic visible encoding to RBM-style latent representation
Below we list the top connections for **every movie** in this subset (not just one example).

In [38]:
hidden_state_x, hidden_state_y = hyperbolic_activation(
    hidden_input["x"],
    hidden_input["y"],
)

hidden_state = {
    "x": hidden_state_x,
    "y": hidden_state_y,
}

print("hidden_state x shape:", hidden_state["x"].shape)
print("hidden_state y shape:", hidden_state["y"].shape)
print("hidden_state x (first 8):", hidden_state["x"][:8])
print("hidden_state y (first 8):", hidden_state["y"][:8])


hidden_state x shape: (64,)
hidden_state y shape: (64,)
hidden_state x (first 8): [ 5.51848476e+04 -1.45628327e+00 -1.88232731e+00 -1.09591561e+00
  1.46018391e+00 -1.24365607e+00 -1.28403659e+00  1.29111477e+00]
hidden_state y (first 8): [-7.52685545e+04  1.05865998e+00  1.59472760e+00  4.48364830e-01
 -1.06401929e+00  7.39378397e-01  8.05450170e-01 -8.16686818e-01]


Activation (Eq. 34) normalizes hyperbolic states onto the unit hyperbola.

In [39]:
print("[Validation] running...")
# Ensure required tensors exist (run Step 3/5 if needed)
if "hidden_input" not in globals() or "hidden_state" not in globals():
    raise RuntimeError("Please run Step 3 and Step 5 cells first.")

input_modulus = hyperbolic_modulus(hidden_input["x"], hidden_input["y"])
state_modulus = hyperbolic_modulus(hidden_state["x"], hidden_state["y"])

print("Hidden input modulus (first 8):", input_modulus[:8])
print("Hidden state modulus (first 8):", state_modulus[:8])

print("hidden_input x (first 8):", hidden_input["x"][:8])
print("hidden_input y (first 8):", hidden_input["y"][:8])

print("hidden_state x (first 8):", hidden_state["x"][:8])
print("hidden_state y (first 8):", hidden_state["y"][:8])

validation_df = pd.DataFrame({
    "input_modulus": input_modulus[:8],
    "state_modulus": state_modulus[:8],
    "input_x": hidden_input["x"][:8],
    "input_y": hidden_input["y"][:8],
    "state_x": hidden_state["x"][:8],
    "state_y": hidden_state["y"][:8],
})
print("[Validation] Preview table:")
display(validation_df)


[Validation] running...
Hidden input modulus (first 8): [1.00000000e-06 4.08747131e-01 2.04264457e-01 1.23112214e-01
 4.98368505e-01 2.31368870e-01 5.21953214e-01 3.31902004e-01]
Hidden state modulus (first 8): [1.e-06 1.e+00 1.e+00 1.e+00 1.e+00 1.e+00 1.e+00 1.e+00]
hidden_input x (first 8): [ 0.05518485 -0.59525161 -0.38449257 -0.1349206   0.72770967 -0.2877433
 -0.67020703  0.42852358]
hidden_input y (first 8): [-0.07526855  0.43272423  0.32574617  0.05519919 -0.5302737   0.17106914
  0.42040731 -0.27105999]
hidden_state x (first 8): [ 5.51848476e+04 -1.45628327e+00 -1.88232731e+00 -1.09591561e+00
  1.46018391e+00 -1.24365607e+00 -1.28403659e+00  1.29111477e+00]
hidden_state y (first 8): [-7.52685545e+04  1.05865998e+00  1.59472760e+00  4.48364830e-01
 -1.06401929e+00  7.39378397e-01  8.05450170e-01 -8.16686818e-01]
[Validation] Preview table:


,input_modulus,state_modulus,input_x,input_y,state_x,state_y
0,0.000001,0.000001,0.055185,-0.075269,55184.847553,-75268.554486
1,0.408747,1.000000,-0.595252,0.432724,-1.456283,1.058660
2,0.204264,1.000000,-0.384493,0.325746,-1.882327,1.594728
3,0.123112,1.000000,-0.134921,0.055199,-1.095916,0.448365
4,0.498369,1.000000,0.727710,-0.530274,1.460184,-1.064019
5,0.231369,1.000000,-0.287743,0.171069,-1.243656,0.739378
6,0.521953,1.000000,-0.670207,0.420407,-1.284037,0.805450
7,0.331902,1.000000,0.428524,-0.271060,1.291115,-0.816687


In [40]:
# Domain validation: admissible states require Re(z) > 0 (x > 0)
violations = hidden_state["x"] <= 0

print("Domain violations:", np.sum(violations))

if np.any(violations):
    print("Example violating indices:", np.where(violations)[0][:10])


Domain violations: 31
Example violating indices: [ 1  2  3  5  6 13 14 15 18 19]


## Step 6 — interpretation in the RBM context

- visible layer = movies
- hidden layer = latent preference factors
- the visible vector is hyperbolically encoded from MovieLens preferences
- the weighted sum computes hidden pre-activation
- the hyperbolic activation produces valid hidden states

This notebook stops at hidden-state computation and does not yet define the energy function.